# 🚘 CoreVision — Brand-Specific Car **Model** Classifier Training

This notebook trains **one model classifier per brand** from:

`/content/drive/MyDrive/CoreVision/data/manual_brands/<brand>/<model>/*.jpg`

Example structure:
- `manual_brands/opel/astra/...`
- `manual_brands/opel/corsa/...`
- `manual_brands/renault/megane/...`
- `manual_brands/renault/clio/...`
- `manual_brands/seat/leon/...`
- `manual_brands/seat/ibiza/...`

This matches your two-stage pipeline:
1. Predict **brand** first
2. Predict **model inside that brand only**


In [ ]:
# ============================================================
# CELL 1: Runtime setup
# ============================================================
!pip install -q timm

import os
import json
import random
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('GPU memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab. Make sure DATA_ROOT points to your dataset path.')


In [ ]:
# ============================================================
# CELL 2: Configuration
# ============================================================
DRIVE_BASE = '/content/drive/MyDrive/CoreVision'
DATA_ROOT = os.path.join(DRIVE_BASE, 'data', 'manual_brands')

# Outputs
WEIGHTS_DIR = os.path.join(DRIVE_BASE, 'weights', 'brand_model_classifiers')
CLASSES_DIR = os.path.join(DRIVE_BASE, 'data', 'brand_model_classes')
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(CLASSES_DIR, exist_ok=True)

# Start with these brands. Set to None to train all brands found in DATA_ROOT.
TARGET_BRANDS = ['opel', 'renault', 'seat']

MIN_IMAGES_PER_MODEL = 8
VAL_RATIO = 0.2
INPUT_SIZE = 224
BATCH_SIZE = 24
NUM_WORKERS = 2

# Two-phase training: fast warmup + fine-tuning
WARMUP_EPOCHS = 2
FINETUNE_EPOCHS = 8
LR_HEAD = 1e-3
LR_ALL = 1e-4
WEIGHT_DECAY = 1e-4
SEED = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DATA_ROOT:', DATA_ROOT)
print('WEIGHTS_DIR:', WEIGHTS_DIR)
print('CLASSES_DIR:', CLASSES_DIR)
print('Device:', DEVICE)


In [ ]:
# ============================================================
# CELL 3: Dataset utilities
# ============================================================
IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')


def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


def collect_brand_samples(brand_dir, min_images=8):
    """Return class names and image paths grouped by class index."""
    model_to_paths = {}

    for model_name in sorted(os.listdir(brand_dir)):
        model_path = os.path.join(brand_dir, model_name)
        if not os.path.isdir(model_path):
            continue

        image_paths = [
            os.path.join(model_path, f)
            for f in os.listdir(model_path)
            if f.lower().endswith(IMAGE_EXTS)
        ]

        if len(image_paths) >= min_images:
            model_to_paths[model_name] = sorted(image_paths)

    class_names = sorted(model_to_paths.keys())
    if len(class_names) < 2:
        raise ValueError(
            f'Need at least 2 model folders with >={min_images} images in: {brand_dir}'
        )

    samples_by_label = {}
    for label, class_name in enumerate(class_names):
        samples_by_label[label] = model_to_paths[class_name]

    return class_names, samples_by_label


def stratified_split(samples_by_label, val_ratio=0.2, seed=42):
    rng = random.Random(seed)
    train_samples = []
    val_samples = []

    for label, paths in samples_by_label.items():
        paths = list(paths)
        rng.shuffle(paths)

        val_count = max(1, int(len(paths) * val_ratio))
        val_count = min(val_count, len(paths) - 1)

        val_paths = paths[:val_count]
        train_paths = paths[val_count:]

        train_samples.extend((p, label) for p in train_paths)
        val_samples.extend((p, label) for p in val_paths)

    rng.shuffle(train_samples)
    rng.shuffle(val_samples)
    return train_samples, val_samples


class ImageListDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)


train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(INPUT_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


In [ ]:
# ============================================================
# CELL 4: Model + training helpers
# ============================================================
def build_model(num_classes, hidden_dim=512):
    backbone = timm.create_model('tf_efficientnetv2_s', pretrained=True, num_classes=0)
    feature_dim = backbone.num_features

    model = nn.Sequential(
        backbone,
        nn.Dropout(0.3),
        nn.Linear(feature_dim, hidden_dim),
        nn.BatchNorm1d(hidden_dim),
        nn.ReLU(inplace=True),
        nn.Dropout(0.2),
        nn.Linear(hidden_dim, num_classes),
    )
    return model


def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)
            if is_train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_count += labels.size(0)

    avg_loss = total_loss / max(total_count, 1)
    avg_acc = total_correct / max(total_count, 1)
    return avg_loss, avg_acc


def train_single_brand(brand_folder):
    brand_key = brand_folder.lower()
    brand_dir = os.path.join(DATA_ROOT, brand_folder)

    class_names, samples_by_label = collect_brand_samples(
        brand_dir,
        min_images=MIN_IMAGES_PER_MODEL,
    )
    train_samples, val_samples = stratified_split(
        samples_by_label,
        val_ratio=VAL_RATIO,
        seed=SEED,
    )

    train_dataset = ImageListDataset(train_samples, transform=train_transform)
    val_dataset = ImageListDataset(val_samples, transform=val_transform)

    pin_memory = torch.cuda.is_available()
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
    )

    print(f'\n=== Training {brand_key} model classifier ===')
    print(f'Classes ({len(class_names)}): {class_names}')
    print(f'Train images: {len(train_dataset)} | Val images: {len(val_dataset)}')

    model = build_model(num_classes=len(class_names), hidden_dim=512).to(DEVICE)
    criterion = nn.CrossEntropyLoss()

    best_acc = -1.0
    best_state = None

    # Phase 1: warmup head only
    for p in model[0].parameters():
        p.requires_grad = False

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_HEAD,
        weight_decay=WEIGHT_DECAY,
    )

    for epoch in range(1, WARMUP_EPOCHS + 1):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
        print(
            f'[Warmup {epoch}/{WARMUP_EPOCHS}] '
            f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} '
            f'val_loss={val_loss:.4f} val_acc={val_acc:.4f}'
        )
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # Phase 2: fine-tune full model
    for p in model[0].parameters():
        p.requires_grad = True

    optimizer = optim.AdamW(
        model.parameters(),
        lr=LR_ALL,
        weight_decay=WEIGHT_DECAY,
    )

    for epoch in range(1, FINETUNE_EPOCHS + 1):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
        print(
            f'[Tune   {epoch}/{FINETUNE_EPOCHS}] '
            f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} '
            f'val_loss={val_loss:.4f} val_acc={val_acc:.4f}'
        )
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)

    weights_path = os.path.join(WEIGHTS_DIR, f'{brand_key}_model_classifier.pth')
    classes_path = os.path.join(CLASSES_DIR, f'{brand_key}_model_classes.txt')

    checkpoint = {
        'model_state_dict': model.state_dict(),
        'accuracy': float(best_acc),
        'class_names': class_names,
        'num_classes': len(class_names),
        'hidden_dim': 512,
        'brand': brand_key,
        'input_size': INPUT_SIZE,
        'trained_at_utc': datetime.utcnow().isoformat() + 'Z',
    }
    torch.save(checkpoint, weights_path)

    with open(classes_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(class_names))

    print(f'Saved weights: {weights_path}')
    print(f'Saved classes: {classes_path}')

    return {
        'brand': brand_key,
        'num_classes': len(class_names),
        'best_val_acc': round(float(best_acc), 4),
        'weights_path': weights_path,
        'classes_path': classes_path,
    }


In [ ]:
# ============================================================
# CELL 5: Train selected brands
# ============================================================
if not os.path.isdir(DATA_ROOT):
    raise FileNotFoundError(f'Dataset root not found: {DATA_ROOT}')

available_brand_dirs = [
    d for d in sorted(os.listdir(DATA_ROOT))
    if os.path.isdir(os.path.join(DATA_ROOT, d))
]
brand_lookup = {d.lower(): d for d in available_brand_dirs}

if TARGET_BRANDS is None:
    selected_keys = sorted(brand_lookup.keys())
else:
    selected_keys = [b.lower() for b in TARGET_BRANDS]

missing = [b for b in selected_keys if b not in brand_lookup]
if missing:
    print('Missing brand folders:', missing)

selected_folders = [brand_lookup[b] for b in selected_keys if b in brand_lookup]
if not selected_folders:
    raise ValueError('No matching brand folders to train.')

results = {}
for folder in selected_folders:
    result = train_single_brand(folder)
    results[result['brand']] = result

manifest_path = os.path.join(CLASSES_DIR, 'brand_model_training_manifest.json')
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)

print('\n=== Training complete ===')
for brand, info in sorted(results.items()):
    print(f"{brand:10s} | classes={info['num_classes']:2d} | best_val_acc={info['best_val_acc']:.4f}")
print('Manifest:', manifest_path)


## Output files

For each trained brand, this notebook writes:
- `weights/brand_model_classifiers/<brand>_model_classifier.pth`
- `data/brand_model_classes/<brand>_model_classes.txt`

When you add a new brand later (for example `cupra/formentor`, `cupra/leon`),
just add the folder and rerun training for that brand.
